# 量子电路 AI 编译器 V11 (DQN 进阶架构) - 专供 Colab 并行号探索
使用异策略的 Deep Q-Network 和经验回放记忆池，专治 PPO 在死胡同图环境中探索崩塌的顽疾。
**注意：请确保上方菜单栏 `代码执行程序` -> `更改运行时类型` 中已选择 `T4 GPU`。**

In [ ]:
# 1. 挂载环境与安装
!pip install -q qiskit==2.3.1 qiskit-aer==0.17.2 gymnasium==1.2.3 rustworkx
!pip install -q torch_geometric
import torch
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html

In [ ]:
# 2. 获取包含 DQN 分支的最新代码
%cd /content
!rm -rf ZJU-Quantum-Compiler
!git clone https://github.com/qqyyqq812/ZJU-Quantum-Compiler.git
%cd /content/ZJU-Quantum-Compiler

In [ ]:
# 3. 云盘挂载
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE = '/content/drive/MyDrive/quantum_train/models/v11_dqn_tokyo20'
!mkdir -p {DRIVE_SAVE}
!mkdir -p models
!ln -s {DRIVE_SAVE} models/v11_dqn_tokyo20

In [ ]:
# 4. 启动 V11 DQN 训练逻辑
import os
os.environ['PYTHONPATH'] = '/content/ZJU-Quantum-Compiler'

from src.compiler.dqn_train import train_dqn
train_dqn(topology_name='ibm_tokyo', n_episodes=50000, save_dir='models/v11_dqn_tokyo20')


In [ ]:
# 5. 📉 训练可视化监控 (V11 DQN)
# (可以在训练进行中手动停止上方格子的运行，运行本格子看图后再重新上滑点运行)
import json, matplotlib.pyplot as plt, numpy as np
import os

DRIVE_SAVE = '/content/drive/MyDrive/quantum_train/models/v11_dqn_tokyo20'
history_file = f'{DRIVE_SAVE}/history_v11_dqn_ibm_tokyo.json'

if os.path.exists(history_file):
    with open(history_file) as f:
        h = json.load(f)
    s = h['episode_swaps']
    window = 100
    avg = [np.mean(s[max(0,i-window):i+1]) for i in range(len(s))]
    plt.figure(figsize=(12,4))
    plt.plot(avg, label='SWAP (Moving Average)', color='orange')
    plt.axhline(y=20, color='r', linestyle='--', label='Target SWAP<=20')
    plt.xlabel('Episode'); plt.ylabel('SWAP count')
    plt.legend(); plt.title(f'V11 Graph-DQN Training Curve (Total: {len(s)} ep)')
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('⚠️ 暂未找到训练日志文件，请稍等训练写入历史后再运行。')